# AIST-FYP Colab: Build Sentence Indexes

This notebook pre-builds sentence-level embedding indexes for both RAGTruth and CiteBench retrieval workflows.

## What this notebook does
1. Mounts Google Drive and clones the repo
2. Installs project dependencies
3. Validates benchmark data paths
4. Builds RAGTruth sentence index artifacts
5. Builds CiteBench sentence index artifacts
6. Verifies output files and mirrors them to Drive

## Dataset Prerequisites

Before running this notebook, ensure:

- `benchmark/RAGTruth/dataset` exists
- `benchmark/CiteEval` oracle data exists (`asqa`, `eli5`, or `msmarco`)
- `data/` is linked to your Drive artifact root

Default output folders:
- `data/indexes/ragtruth_sentences/{RAGTRUTH_SPLIT}`
- `data/indexes/citeeval_sentences/{CITEEVAL_ORACLE_DATASET}`

### Configuration

This notebook builds sentence indexes in two explicit passes:

1. RAGTruth sentence index build
2. CiteBench sentence index build

Both outputs are written to separate directories so they can coexist safely.

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["evaluation"]

ENCODER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DEVICE = "cuda"
BATCH_SIZE = 32

# Build targets
RAGTRUTH_SPLIT = "test"  # test | train | all
CITEEVAL_ORACLE_DATASET = "asqa"  # asqa | eli5 | msmarco

# Optional output overrides (empty = use defaults)
RAGTRUTH_OUTPUT_DIR_OVERRIDE = ""
CITEEVAL_OUTPUT_DIR_OVERRIDE = ""

DRIVE_DATA_ROOT = "/content/drive/MyDrive/data"
DRIVE_RAGTRUTH_DATASET_ROOT = "/content/drive/MyDrive/AIST-FYP/benchmark/RAGTruth/dataset"
DRIVE_CITEEVAL_ROOT = "/content/drive/MyDrive/AIST-FYP/benchmark/CiteEval"

# Build artifact persistence root
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIST-FYP-colab-preprocess"
DRIVE_WORK_ROOT = f"{DRIVE_OUTPUT_DIR}/work_eval"
LOCAL_WORK_ROOT = DRIVE_WORK_ROOT


In [ ]:
import os
import json
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

def _normalized_unique_paths(entries):
    ordered = []
    seen = set()
    for entry in entries:
        if not entry:
            continue
        resolved = str(Path(entry).resolve())
        if resolved not in seen:
            ordered.append(resolved)
            seen.add(resolved)
    return ordered

def run(cmd, cwd=None, check=True, stream=True):
    env = os.environ.copy()
    repo_path = str(Path(cwd).resolve()) if cwd else str(Path(os.getcwd()).resolve())

    existing_pp = env.get('PYTHONPATH', '').split(os.pathsep)
    pp_entries = _normalized_unique_paths([repo_path, *existing_pp])

    site_pkgs = [p for p in sys.path if 'site-packages' in p]
    pp_entries = _normalized_unique_paths([*pp_entries, *site_pkgs])

    env['PYTHONPATH'] = os.pathsep.join(pp_entries)

    print(f"\n$ {cmd}")

    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            executable='/bin/bash',
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout="".join(out_lines),
            stderr=None,
        )
    else:
        completed = subprocess.run(
            cmd,
            shell=True,
            cwd=cwd,
            env=env,
            text=True,
            capture_output=True,
            executable='/bin/bash',
        )
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def ensure_symlink_dir(link_path: Path, target_path: Path):
    target_path = Path(target_path)
    link_path = Path(link_path)
    ensure_dir(target_path)
    ensure_dir(link_path.parent)

    if link_path.is_symlink():
        current_target = Path(os.readlink(link_path))
        if current_target == target_path:
            return
        link_path.unlink()
    elif link_path.exists():
        if link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()

    os.symlink(target_path, link_path, target_is_directory=True)

def copytree_merge(src, dst):
    src_p = Path(src)
    dst_p = Path(dst)
    if not src_p.exists():
        return
    for p in src_p.rglob('*'):
        rel = p.relative_to(src_p)
        t = dst_p / rel
        if p.is_dir():
            t.mkdir(parents=True, exist_ok=True)
        else:
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, t)

def exists_or_raise(path, msg):
    if not Path(path).exists():
        raise FileNotFoundError(f"{msg}: {path}")


In [ ]:
# Override helper for filesystems (e.g., Google Drive FUSE) that do not support symlinks.
def ensure_symlink_dir(link_path: Path, target_path: Path):
    target_path = Path(target_path)
    link_path = Path(link_path)
    ensure_dir(target_path)
    ensure_dir(link_path.parent)

    if link_path.is_symlink():
        current_target = Path(os.readlink(link_path))
        if current_target == target_path:
            return
        link_path.unlink()
    elif link_path.exists():
        if link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()

    try:
        os.symlink(target_path, link_path, target_is_directory=True)
    except OSError as e:
        if getattr(e, 'errno', None) == 95 or 'Operation not supported' in str(e):
            ensure_dir(link_path)
            print(f"Symlink not supported; using real directory instead: {link_path}")
            return
        raise


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

if not str(LOCAL_WORK_ROOT).startswith('/content/drive/'):
    print(f"WARN: LOCAL_WORK_ROOT is not on Drive: {LOCAL_WORK_ROOT}")
    print("Artifacts may not survive Colab runtime reset.")

ensure_dir(LOCAL_WORK_ROOT)
print("Persistent LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)

# Clone/update repo
if Path(REPO_DIR).exists() and (Path(REPO_DIR) / '.git').exists():
    print(f"Repo dir already exists: {REPO_DIR}")
    run("git fetch --all", cwd=REPO_DIR, check=False)
    run(f"git checkout {REPO_BRANCH}", cwd=REPO_DIR, check=False)
    run(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR, check=False)
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)

# Ensure repo root is first for imports.
repo_root = str(Path(REPO_DIR).resolve())
existing_pp = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
ordered_pp = []
seen = set()
for entry in [repo_root, *existing_pp]:
    resolved = str(Path(entry).resolve())
    if resolved not in seen:
        ordered_pp.append(resolved)
        seen.add(resolved)
os.environ['PYTHONPATH'] = os.pathsep.join(ordered_pp)

print('PYTHONPATH (repo-first):', os.environ['PYTHONPATH'])
print('src/utils/config.py exists:', (Path(REPO_DIR) / 'src' / 'utils' / 'config.py').exists())

# Link project data path to Drive data artifacts root
project_data_path = Path(REPO_DIR) / 'data'
drive_data_root = Path(DRIVE_DATA_ROOT)
if not drive_data_root.exists():
    raise FileNotFoundError(f"Drive data root not found: {drive_data_root}")
ensure_symlink_dir(project_data_path, drive_data_root)
print(f"Data artifact path: {project_data_path} -> {project_data_path.resolve()}")

# Link benchmark roots to Drive if available
project_ragtruth_dataset = Path(REPO_DIR) / 'benchmark' / 'RAGTruth' / 'dataset'
if Path(DRIVE_RAGTRUTH_DATASET_ROOT).exists():
    ensure_symlink_dir(project_ragtruth_dataset, Path(DRIVE_RAGTRUTH_DATASET_ROOT))
    print(f"RAGTruth dataset path: {project_ragtruth_dataset} -> {project_ragtruth_dataset.resolve()}")
else:
    print(f"WARN: RAGTruth dataset root missing at {DRIVE_RAGTRUTH_DATASET_ROOT}")

project_citeeval_root = Path(REPO_DIR) / 'benchmark' / 'CiteEval'
if Path(DRIVE_CITEEVAL_ROOT).exists():
    ensure_symlink_dir(project_citeeval_root, Path(DRIVE_CITEEVAL_ROOT))
    print(f"CiteEval benchmark path: {project_citeeval_root} -> {project_citeeval_root.resolve()}")
else:
    print(f"WARN: CiteEval root missing at {DRIVE_CITEEVAL_ROOT}")


In [ ]:
# Install dependencies
kernel_python = sys.executable
print('Notebook Python executable:', kernel_python)
run(f'"{kernel_python}" -m pip install -U pip wheel setuptools', stream=True)
run(f'"{kernel_python}" -m pip install -U faiss-cpu rank_bm25', stream=True)
run(f'"{kernel_python}" -m pip install -U uv', stream=True)

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False, stream=True)

if result.returncode == 0:
    uv_python = uv_project / '.venv' / 'bin' / 'python'
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    run(f"{uv_python} - <<\"PY\"\nimport sys\nprint('uv python executable:', sys.executable)\nPY", cwd=REPO_DIR)
    print(f"uv sync complete: {uv_project}")
else:
    print("WARN: uv sync failed, using pip requirements fallback")
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f'"{kernel_python}" -m pip install --extra-index-url {pytorch_index} -r {requirements_path}'
    run(install_cmd, cwd=REPO_DIR, check=False, stream=True)


In [ ]:
# Runtime sanity checks
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

repo_root = str(Path(REPO_DIR).resolve())
existing_pp = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
ordered_pp = []
seen = set()
for entry in [repo_root, *existing_pp]:
    resolved = str(Path(entry).resolve())
    if resolved not in seen:
        ordered_pp.append(resolved)
        seen.add(resolved)
os.environ['PYTHONPATH'] = os.pathsep.join(ordered_pp)

print('PYTHONPATH (top 6):', ordered_pp[:6])


In [ ]:
# Preflight: RAGTruth sentence index build
repo = Path(REPO_DIR)
ragtruth_dataset = repo / 'benchmark' / 'RAGTruth' / 'dataset'
exists_or_raise(ragtruth_dataset, 'Missing RAGTruth dataset directory')

if RAGTRUTH_OUTPUT_DIR_OVERRIDE.strip():
    ragtruth_output_dir = RAGTRUTH_OUTPUT_DIR_OVERRIDE.strip()
else:
    ragtruth_output_dir = f"data/indexes/ragtruth_sentences/{RAGTRUTH_SPLIT}"

print('RAGTruth preflight passed')
print('RAGTruth output dir:', ragtruth_output_dir)


In [ ]:
# Build command: RAGTruth
repo = Path(REPO_DIR).resolve()

ragtruth_build_cmd = (
    f'"{sys.executable}" scripts/build_sentence_index.py '
    f'--dataset ragtruth --split {RAGTRUTH_SPLIT} '
    f'--encoder "{ENCODER_MODEL}" --device {DEVICE} --batch-size {BATCH_SIZE} '
    f'--output-dir {ragtruth_output_dir}'
)

print('RAGTruth build command:')
print(ragtruth_build_cmd)
run(ragtruth_build_cmd, cwd=str(repo))
print('RAGTruth sentence index build completed.')


In [ ]:
# Verify and mirror: RAGTruth
ragtruth_out_path = (Path(REPO_DIR) / ragtruth_output_dir).resolve()
ragtruth_required_files = [
    ragtruth_out_path / 'sentences.jsonl',
    ragtruth_out_path / 'embeddings.npy',
    ragtruth_out_path / 'sample_index.json',
]
for file_path in ragtruth_required_files:
    exists_or_raise(file_path, 'Missing RAGTruth sentence-index artifact')
    print('Found:', file_path)

ragtruth_mirror_dir = Path(DRIVE_OUTPUT_DIR) / 'sentence_indexes' / 'ragtruth' / RAGTRUTH_SPLIT
ensure_dir(ragtruth_mirror_dir)
for file_path in ragtruth_required_files:
    shutil.copy2(file_path, ragtruth_mirror_dir / file_path.name)

print('RAGTruth artifacts mirrored to:', ragtruth_mirror_dir)


In [ ]:
# Preflight: CiteBench sentence index build
repo = Path(REPO_DIR)
citeeval_root = repo / 'benchmark' / 'CiteEval'
exists_or_raise(citeeval_root, 'Missing CiteEval benchmark directory')

if CITEEVAL_OUTPUT_DIR_OVERRIDE.strip():
    citeeval_output_dir = CITEEVAL_OUTPUT_DIR_OVERRIDE.strip()
else:
    citeeval_output_dir = f"data/indexes/citeeval_sentences/{CITEEVAL_ORACLE_DATASET}"

print('CiteBench preflight passed')
print('CiteBench output dir:', citeeval_output_dir)


In [ ]:
# Build command: CiteBench
repo = Path(REPO_DIR).resolve()

citeeval_build_cmd = (
    f'"{sys.executable}" scripts/build_sentence_index.py '
    f'--dataset citeeval --oracle-dataset {CITEEVAL_ORACLE_DATASET} '
    f'--encoder "{ENCODER_MODEL}" --device {DEVICE} --batch-size {BATCH_SIZE} '
    f'--output-dir {citeeval_output_dir}'
)

print('CiteBench build command:')
print(citeeval_build_cmd)
run(citeeval_build_cmd, cwd=str(repo))
print('CiteBench sentence index build completed.')


In [ ]:
# Verify and mirror: CiteBench
citeeval_out_path = (Path(REPO_DIR) / citeeval_output_dir).resolve()
citeeval_required_files = [
    citeeval_out_path / 'sentences.jsonl',
    citeeval_out_path / 'embeddings.npy',
    citeeval_out_path / 'sample_index.json',
]
for file_path in citeeval_required_files:
    exists_or_raise(file_path, 'Missing CiteBench sentence-index artifact')
    print('Found:', file_path)

citeeval_mirror_dir = Path(DRIVE_OUTPUT_DIR) / 'sentence_indexes' / 'citebench' / CITEEVAL_ORACLE_DATASET
ensure_dir(citeeval_mirror_dir)
for file_path in citeeval_required_files:
    shutil.copy2(file_path, citeeval_mirror_dir / file_path.name)

print('CiteBench artifacts mirrored to:', citeeval_mirror_dir)


## Run Order and Artifact Notes

Recommended execution order:
1. Mount/clone/setup cell
2. Dependency install cell
3. Runtime sanity cell
4. RAGTruth preflight -> build -> verify cells
5. CiteBench preflight -> build -> verify cells

This notebook builds two independent sentence indexes that can coexist:
- `data/indexes/ragtruth_sentences/{RAGTRUTH_SPLIT}`
- `data/indexes/citeeval_sentences/{CITEEVAL_ORACLE_DATASET}`

Each index directory contains:
- `sentences.jsonl`
- `embeddings.npy`
- `sample_index.json`

Mirrored copies are stored under:
- `My Drive/AIST-FYP-colab-preprocess/sentence_indexes/ragtruth/{split}`
- `My Drive/AIST-FYP-colab-preprocess/sentence_indexes/citebench/{oracle_dataset}`
